# 45 — Pairwise Ranking LGBM

Exploit cliff pair ordering: train a cliff-direction classifier on difference vectors, then apply a pairwise correction on top of the standard LGBM regression predictions.

**Primary metric:** RAE (lower is better). Current best OOF RAE: 0.5281.

In [1]:
import sys, os
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                   reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load data + cliff pairs

In [2]:
tr = load_train()
te = load_test()
print(f"Train: {len(tr)} | Test: {len(te)}")

# Compute scaffolds for CV splitting
tr['scaffold'] = tr['smiles'].apply(bemis_murcko)

# Features
X_tr_raw = combined(tr['smiles'].tolist())
X_te_raw = combined(te['smiles'].tolist())
X_tr = impute(X_tr_raw)
X_te = impute(X_te_raw)
y_tr = tr['pec50'].values.astype(np.float32)
print(f"X_tr: {X_tr.shape} | X_te: {X_te.shape}")

Train: 4139 | Test: 513


X_tr: (4139, 2265) | X_te: (513, 2265)


In [3]:
# Load or compute cliff labels
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
if cliff_path.exists():
    cliff_labels = pd.read_parquet(cliff_path)
    tr = tr.merge(cliff_labels[['name','is_cliff_member','cliff_role','max_cliff_delta']],
                  on='name', how='left')
    tr['is_cliff_member'] = tr['is_cliff_member'].fillna(False)
    tr['cliff_role'] = tr['cliff_role'].fillna(0)
else:
    # Vectorized Tanimoto computation
    fps = morgan_fp_batch(tr.smiles.tolist()).astype(np.float32)  # (N, 2048)
    dot = fps @ fps.T
    rowsum = fps.sum(1)
    union = rowsum[:, None] + rowsum[None, :] - dot
    tanimoto_mat = dot / union.clip(min=1)
    np.fill_diagonal(tanimoto_mat, 0)
    tr['is_cliff_member'] = False
    tr['cliff_role'] = 0
    tr['max_cliff_delta'] = 0.0
    pec50_arr = tr['pec50'].values
    for i in range(len(tr)):
        for j in range(i+1, len(tr)):
            if tanimoto_mat[i, j] >= 0.6 and abs(pec50_arr[i] - pec50_arr[j]) >= 1.0:
                delta = abs(pec50_arr[i] - pec50_arr[j])
                tr.loc[tr.index[i], 'is_cliff_member'] = True
                tr.loc[tr.index[j], 'is_cliff_member'] = True
                if pec50_arr[i] > pec50_arr[j]:
                    tr.loc[tr.index[i], 'cliff_role'] = 1
                    tr.loc[tr.index[j], 'cliff_role'] = -1
                else:
                    tr.loc[tr.index[i], 'cliff_role'] = -1
                    tr.loc[tr.index[j], 'cliff_role'] = 1
                tr.loc[tr.index[i], 'max_cliff_delta'] = max(
                    tr.loc[tr.index[i], 'max_cliff_delta'], delta)
                tr.loc[tr.index[j], 'max_cliff_delta'] = max(
                    tr.loc[tr.index[j], 'max_cliff_delta'], delta)

print(f"Cliff members in train: {tr['is_cliff_member'].sum()}")

Cliff members in train: 62


In [4]:
# Load or compute cliff pairs
pairs_path = DATA_PROCESSED / 'cliff_pairs.parquet'
if pairs_path.exists():
    cliff_pairs_df = pd.read_parquet(pairs_path)
    print(f"Loaded {len(cliff_pairs_df)} cliff pairs from disk")
else:
    # Vectorized Tanimoto
    fps = morgan_fp_batch(tr.smiles.tolist()).astype(np.float32)
    dot = fps @ fps.T
    rowsum = fps.sum(1)
    union = rowsum[:, None] + rowsum[None, :] - dot
    tanimoto_mat = dot / union.clip(min=1)
    np.fill_diagonal(tanimoto_mat, 0)

    pec50_arr = tr['pec50'].values
    pairs = []
    for i in range(len(tr)):
        for j in range(i+1, len(tr)):
            sim = tanimoto_mat[i, j]
            delta = pec50_arr[i] - pec50_arr[j]
            if sim >= 0.6 and abs(delta) >= 1.0:
                if delta > 0:
                    pairs.append({'idx_active': i, 'idx_inactive': j,
                                  'tanimoto': sim, 'delta_pec50': delta})
                else:
                    pairs.append({'idx_active': j, 'idx_inactive': i,
                                  'tanimoto': sim, 'delta_pec50': -delta})
    cliff_pairs_df = pd.DataFrame(pairs)
    print(f"Computed {len(cliff_pairs_df)} cliff pairs inline")

print(cliff_pairs_df.head())

Computed 34 cliff pairs inline
   idx_active  idx_inactive  tanimoto  delta_pec50
0         735          2082  0.689655         2.72
1        1595          1689  0.625000         1.41
2        1647          1623  0.718750         1.49
3        1649          1697  1.000000         1.22
4        1651          3154  0.688889         1.76


## 2. Pairwise augmentation — cliff-direction classifier

In [5]:
def build_pairwise_dataset(X, pairs_df):
    """Build difference-vector dataset for cliff-direction classifier.

    For each pair (active, inactive): X_diff = X_active - X_inactive.
    Label y=1 means active > inactive. Also include reversed pairs with y=0
    for balanced training.
    """
    if len(pairs_df) == 0:
        d = X.shape[1]
        return np.zeros((0, d), dtype=np.float32), np.zeros(0, dtype=np.int32)

    X_diffs, y_labels = [], []
    for _, row in pairs_df.iterrows():
        ia = int(row['idx_active'])
        ii = int(row['idx_inactive'])
        diff = X[ia] - X[ii]
        X_diffs.append(diff)
        y_labels.append(1)
        # Reversed pair — active becomes inactive
        X_diffs.append(-diff)
        y_labels.append(0)

    return np.array(X_diffs, dtype=np.float32), np.array(y_labels, dtype=np.int32)


if len(cliff_pairs_df) > 0:
    X_diff_all, y_diff_all = build_pairwise_dataset(X_tr, cliff_pairs_df)
    print(f"Pairwise dataset: {X_diff_all.shape}, label balance: {y_diff_all.mean():.2f}")
else:
    print("No cliff pairs found — pairwise correction will be skipped")
    X_diff_all = np.zeros((0, X_tr.shape[1]), dtype=np.float32)
    y_diff_all = np.zeros(0, dtype=np.int32)

Pairwise dataset: (68, 2265), label balance: 0.50


## 3. Pairwise correction layer

In [6]:
def apply_pairwise_correction(
    X_query,           # (M, D) features of compounds to correct
    preds_query,       # (M,) initial predictions
    X_train,           # (N, D) training features
    y_train,           # (N,) training labels
    cliff_classifier,  # fitted classifier with predict_proba
    fps_train,         # (N, 2048) uint8 for Tanimoto
    fps_query,         # (M, 2048) uint8 for Tanimoto
    train_is_cliff,    # (N,) bool mask of cliff-member training compounds
    min_tanimoto=0.4,
):
    """Nudge predictions for test compounds near cliff members."""
    corrected = preds_query.copy()
    cliff_train_idx = np.where(train_is_cliff)[0]

    if len(cliff_train_idx) == 0 or cliff_classifier is None:
        return corrected

    fps_cliff = fps_train[cliff_train_idx].astype(np.float32)
    fps_q = fps_query.astype(np.float32)

    # Tanimoto between query and cliff-member training compounds
    dot = fps_q @ fps_cliff.T
    rowsum_q = fps_q.sum(1)
    rowsum_c = fps_cliff.sum(1)
    union = rowsum_q[:, None] + rowsum_c[None, :] - dot
    tanimoto_qc = dot / union.clip(min=1)  # (M, n_cliff)

    for m in range(len(X_query)):
        best_cliff_pos = int(np.argmax(tanimoto_qc[m]))
        best_sim = tanimoto_qc[m, best_cliff_pos]

        if best_sim < min_tanimoto:
            continue

        train_idx_cliff = cliff_train_idx[best_cliff_pos]
        neighbor_pec50 = y_train[train_idx_cliff]
        neighbor_x = X_train[train_idx_cliff]

        # Difference vector: query minus neighbor
        diff_vec = X_query[m] - neighbor_x
        prob_active = cliff_classifier.predict_proba(diff_vec.reshape(1, -1))[0, 1]

        # Expected cliff delta ~1.5 log units
        expected_delta = 1.5
        blend_w = min(0.4, (best_sim - 0.3) * 2.0)

        # If prob_active > 0.5 → query should be ABOVE neighbor
        # If prob_active < 0.5 → query should be BELOW neighbor
        direction = 1.0 if prob_active > 0.5 else -1.0
        nudge = direction * expected_delta * blend_w * abs(prob_active - 0.5) * 2.0
        corrected[m] += nudge

    return corrected

print("Pairwise correction function defined.")

Pairwise correction function defined.


## 4. Scaffold 5-fold CV with correction

In [7]:
from sklearn.linear_model import LogisticRegression

splits = scaffold_kfold_indices(tr['scaffold'], n_splits=5)

fps_all = morgan_fp_batch(tr['smiles'].tolist())  # (N, 2048) uint8
tr_is_cliff = tr['is_cliff_member'].values.astype(bool)

oof_preds_base = np.full(len(tr), np.nan)
oof_preds_corrected = np.full(len(tr), np.nan)

fold_results = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    print(f"\nFold {fold+1}/5 — train={len(tr_idx)}, val={len(val_idx)}")

    X_fold_tr, X_fold_val = X_tr[tr_idx], X_tr[val_idx]
    y_fold_tr, y_fold_val = y_tr[tr_idx], y_tr[val_idx]
    fps_fold_tr = fps_all[tr_idx]
    fps_fold_val = fps_all[val_idx]
    is_cliff_fold_tr = tr_is_cliff[tr_idx]

    # Identify cliff pairs within this fold's training set
    fold_tr_set = set(tr_idx)
    if len(cliff_pairs_df) > 0:
        fold_pairs = cliff_pairs_df[
            cliff_pairs_df['idx_active'].isin(fold_tr_set) &
            cliff_pairs_df['idx_inactive'].isin(fold_tr_set)
        ].copy()
        # Re-index to fold-local indices
        idx_map = {orig: local for local, orig in enumerate(tr_idx)}
        fold_pairs['idx_active'] = fold_pairs['idx_active'].map(idx_map)
        fold_pairs['idx_inactive'] = fold_pairs['idx_inactive'].map(idx_map)
        fold_pairs = fold_pairs.dropna(subset=['idx_active', 'idx_inactive'])
        fold_pairs = fold_pairs.astype({'idx_active': int, 'idx_inactive': int})
    else:
        fold_pairs = pd.DataFrame(columns=['idx_active', 'idx_inactive', 'tanimoto', 'delta_pec50'])

    # Train main LGBM regressor
    reg_model = lgb.LGBMRegressor(**LGBM_PARAMS)
    reg_model.fit(X_fold_tr, y_fold_tr)
    base_preds = reg_model.predict(X_fold_val)
    oof_preds_base[val_idx] = base_preds

    rae_base = rae(y_fold_val, base_preds)
    print(f"  Base RAE: {rae_base:.4f}")

    # Train cliff classifier on difference vectors
    cliff_clf = None
    if len(fold_pairs) >= 4:
        X_diff_fold, y_diff_fold = build_pairwise_dataset(X_fold_tr, fold_pairs)
        if len(np.unique(y_diff_fold)) == 2:
            clf_params = dict(C=0.1, max_iter=500, class_weight='balanced',
                              solver='lbfgs', n_jobs=1)
            cliff_clf = LogisticRegression(**clf_params)
            try:
                cliff_clf.fit(X_diff_fold, y_diff_fold)
            except Exception as e:
                print(f"  Cliff classifier failed: {e}")
                cliff_clf = None

    # Apply pairwise correction
    corrected = apply_pairwise_correction(
        X_query=X_fold_val,
        preds_query=base_preds,
        X_train=X_fold_tr,
        y_train=y_fold_tr,
        cliff_classifier=cliff_clf,
        fps_train=fps_fold_tr,
        fps_query=fps_fold_val,
        train_is_cliff=is_cliff_fold_tr,
    )
    oof_preds_corrected[val_idx] = corrected

    rae_corr = rae(y_fold_val, corrected)
    print(f"  Corrected RAE: {rae_corr:.4f}")
    fold_results.append({'fold': fold+1, 'rae_base': rae_base, 'rae_corrected': rae_corr})

results_df = pd.DataFrame(fold_results)
print("\n=== Fold Summary ===")
print(results_df.to_string(index=False))
print(f"\nOOF base RAE:      {rae(y_tr, oof_preds_base):.4f}")
print(f"OOF corrected RAE: {rae(y_tr, oof_preds_corrected):.4f}")


Fold 1/5 — train=3311, val=828


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Base RAE: 0.4934


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Corrected RAE: 0.4986

Fold 2/5 — train=3311, val=828


...

  Corrected RAE: 0.5650

Fold 5/5 — train=3312, val=827


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Base RAE: 0.5952


  Corrected RAE: 0.6016

=== Fold Summary ===
 fold  rae_base  rae_corrected
    1  0.493352       0.498603
    2  0.576243       0.577303
    3  0.596056       0.596138
    4  0.563463       0.564979
    5  0.595235       0.601641

OOF base RAE:      0.5600
OOF corrected RAE: 0.5629


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 5. Final model — train on all data, predict test

In [8]:
# Train final regressor on all training data
final_reg = lgb.LGBMRegressor(**LGBM_PARAMS)
final_reg.fit(X_tr, y_tr)
test_preds_base = final_reg.predict(X_te)

# Train final cliff classifier on all data
final_clf = None
if len(X_diff_all) >= 4 and len(np.unique(y_diff_all)) == 2:
    from sklearn.linear_model import LogisticRegression
    final_clf = LogisticRegression(C=0.1, max_iter=500, class_weight='balanced',
                                   solver='lbfgs', n_jobs=1)
    try:
        final_clf.fit(X_diff_all, y_diff_all)
        print("Final cliff classifier trained.")
    except Exception as e:
        print(f"Final cliff classifier failed: {e}")
        final_clf = None

fps_te = morgan_fp_batch(te['smiles'].tolist())
test_preds_corrected = apply_pairwise_correction(
    X_query=X_te,
    preds_query=test_preds_base,
    X_train=X_tr,
    y_train=y_tr,
    cliff_classifier=final_clf,
    fps_train=fps_all,
    fps_query=fps_te,
    train_is_cliff=tr_is_cliff,
)

# Clip to training range ± 0.5
y_lo = y_tr.min() - 0.5
y_hi = y_tr.max() + 0.5
test_preds_corrected = np.clip(test_preds_corrected, y_lo, y_hi)

print(f"Test pred range: [{test_preds_corrected.min():.3f}, {test_preds_corrected.max():.3f}]")

D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Final cliff classifier trained.
Test pred range: [2.209, 6.052]


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
# Save OOF predictions
np.save(DATA_PROCESSED / 'oof_pairwise_ranking.npy', oof_preds_corrected)
print("Saved OOF predictions.")

# Save submission
sub = pd.DataFrame({'Molecule Name': te['name'], 'pEC50': test_preds_corrected})
out_path = SUBMISSIONS / '45_pairwise_ranking_lgbm.csv'
sub.to_csv(out_path, index=False)
print(f"Saved submission to {out_path}")
print(sub.head())
print(f"\nFinal OOF corrected RAE: {rae(y_tr, oof_preds_corrected):.4f}")

Saved OOF predictions.
Saved submission to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\45_pairwise_ranking_lgbm.csv
    Molecule Name     pEC50
0  OADMET-0006617  4.488582
1  OADMET-0006616  3.355308
2  OADMET-0006615  5.395473
3  OADMET-0006614  5.282305
4  OADMET-0006613  4.799671

Final OOF corrected RAE: 0.5629
